# Transformer TimeVAE Context Forecasting

<!-- Commented sorted notebook -->
Purpose: train and evaluate the cleaned supervisor-facing version of this generative forecaster. This version receives past kWh context plus known future/static covariates.

The notebook follows the shared experiment structure: setup, preprocessing, batching, model training, checkpointing, validation sampling, and evaluation export.


In [ ]:

import math
import os
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", message=r".*'H' is deprecated.*", category=FutureWarning)

MODEL_LABEL = "Transformer-TimeVAE-Forecaster"
MODEL_TAG = "forecast-transformer-timevae"

HOUSEHOLD_ID_COL = "CUSTOMER_KEY"
TIME_COL = "READING_DATETIME"
TARGET_COL = "kWh"

FORECAST_HORIZONS = {"24h": pd.Timedelta("24h"), "7d": pd.Timedelta("7d"), "28d": pd.Timedelta("28d")}
ALLOWED_FREQS = {"30min", "1H", "2H", "3H", "6H"}
FREQ = "30min"
PRIMARY_HORIZON = "24h"

CONTEXT_HORIZONS = {"24h": pd.Timedelta("2d"), "7d": pd.Timedelta("7d"), "28d": pd.Timedelta("7d")}
WINDOW_STRIDE_MULTIPLIER = 1

SEED = 0
VAL_FRAC = 0.20
BATCH_SIZE = 64
LR = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


epochs_vae = 500
hidden_dim = 128
cond_dim = 32
latent_dim = 32
grad_clip = 1.0

beta = 0.015
kl_warmup_epochs = 350


transformer_layers = 2
transformer_heads = 4
transformer_ff_dim = 256
transformer_dropout = 0.10

sample_temperature = 0.8



STATIC_COLS = [
    "NUM_OCCUPANTS", "NUM_ROOMS_HEATED", "NUM_REFRIGERATORS",
    "Unit", "SemiDetached", "SeparateHouse",
    "HAS_GAS_HEATING", "HAS_GAS_HOT_WATER", "HAS_GAS_COOKING",
    "HAS_POOLPUMP", "Ducted", "SplitSystem", "NoAirCon", "OtherAirCon",
    "CONTROLLED_LOAD_CNT",
]

CATEGORICAL_COLS = ["StationNo", "TRIAL_REGION_NAME"]

RAW_TIME_COV_COLS = [
    "Temperature",
    "Weekday",
    "Month",
    "Hour",
    "CDD",
    "HDD",
    "wind_speed",
    "temperature_lag_2",
    "temperature_lag_6",
    "temperature_lag_48",
    "temperature_lag_144",
]

TIME_COV_COLS = [
    "Temperature",
    "Hour_sin",
    "Hour_cos",
    "Weekday_sin",
    "Weekday_cos",
    "Month_sin",
    "Month_cos",
    "CDD",
    "HDD",
    "wind_speed",
    "temperature_lag_2",
    "temperature_lag_6",
    "temperature_lag_48",
    "temperature_lag_144",
]

# Keeps pandas frequency strings explicit so 30min/1H/2H/3H experiments resample consistently.
def pandas_freq(freq: str) -> str:
    return freq.replace("H", "h")

# Converts a human horizon label into target-window steps at the selected data frequency.
def seq_len_for_horizon(primary_horizon, freq):
    if primary_horizon not in FORECAST_HORIZONS:
        raise ValueError(f"Unsupported horizon {primary_horizon}. Use one of {sorted(FORECAST_HORIZONS)}")
    if freq not in ALLOWED_FREQS:
        raise ValueError(f"Unsupported freq {freq}. Use one of {sorted(ALLOWED_FREQS)}")
    steps = FORECAST_HORIZONS[primary_horizon] / pd.to_timedelta(pandas_freq(freq))
    if not float(steps).is_integer():
        raise ValueError(f"Horizon {primary_horizon} is not divisible by freq {freq}")
    return int(steps)

# Uses fixed calendar-duration context windows so context length is comparable across frequencies.
def context_len_for_horizon(primary_horizon, freq):
    if primary_horizon not in CONTEXT_HORIZONS:
        raise ValueError(f"Unsupported context horizon {primary_horizon}. Use one of {sorted(CONTEXT_HORIZONS)}")
    steps = CONTEXT_HORIZONS[primary_horizon] / pd.to_timedelta(pandas_freq(freq))
    if not float(steps).is_integer():
        raise ValueError(f"Context horizon {primary_horizon} is not divisible by freq {freq}")
    return int(steps)

SEQ_LEN = seq_len_for_horizon(PRIMARY_HORIZON, FREQ)
CONTEXT_LEN = context_len_for_horizon(PRIMARY_HORIZON, FREQ)
WINDOW_STRIDE = max(1, WINDOW_STRIDE_MULTIPLIER * SEQ_LEN)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("medium")

print(f"{MODEL_LABEL} | FREQ={FREQ} | horizon={PRIMARY_HORIZON} | context={CONTEXT_LEN} | horizon_steps={SEQ_LEN} | stride={WINDOW_STRIDE} | device={DEVICE}")

In [ ]:

# Searches nearby project folders for the cleaned household energy dataset used by all notebooks.
def find_data_file():
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([
            base / "data_with_weather.pickle",
            base / "models" / "data_with_weather.pickle",
            base / "data_min.parquet",
            base / "models" / "data_min.parquet",
        ])
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find data_with_weather.pickle or data_min.parquet from the notebook location.")

# Cyclical sin/cos encoding preserves periodic hour/weekday/month distance; source: https://feature-engine.trainindata.com/en/latest/user_guide/creation/CyclicalFeatures.html
def add_cyclical_time_features(frame, datetime_values):
    dt = pd.DatetimeIndex(pd.to_datetime(datetime_values))
    hour = dt.hour.astype(np.float32)
    minute = dt.minute.astype(np.float32)
    hour_float = hour + minute / 60.0
    weekday = dt.weekday.astype(np.float32)
    month = dt.month.astype(np.float32)

    frame["Hour_sin"] = np.sin(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Hour_cos"] = np.cos(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Weekday_sin"] = np.sin(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Weekday_cos"] = np.cos(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Month_sin"] = np.sin(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Month_cos"] = np.cos(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Hour"] = hour.astype("float32")
    frame["Weekday"] = weekday.astype("float32")
    frame["Month"] = month.astype("float32")
    return frame

data_path = find_data_file()
if data_path.suffix == ".pickle":
    df = pd.read_pickle(data_path)
else:
    df = pd.read_parquet(data_path)

needed = [TIME_COL, TARGET_COL, HOUSEHOLD_ID_COL] + STATIC_COLS + CATEGORICAL_COLS + RAW_TIME_COV_COLS
for col in needed:
    if col not in df.columns:
        df[col] = 0.0 if col not in CATEGORICAL_COLS else "missing"
df = df[needed].copy()

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df[HOUSEHOLD_ID_COL] = pd.to_numeric(df[HOUSEHOLD_ID_COL], errors="coerce").astype("int64")
df = df.sort_values([HOUSEHOLD_ID_COL, TIME_COL])
add_cyclical_time_features(df, df[TIME_COL])

for col in STATIC_COLS + RAW_TIME_COV_COLS + [TARGET_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).astype("float32")

if "Temperature" in df.columns:
    temp = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0)
    df["CDD"] = np.maximum(temp - 18.0, 0.0).astype("float32")
    df["HDD"] = np.maximum(18.0 - temp, 0.0).astype("float32")

for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("string").fillna("missing")
    df[f"{col}_id"], _ = pd.factorize(df[col], sort=True)
    df[f"{col}_id"] = df[f"{col}_id"].astype("int64")

num_stations = int(df["StationNo_id"].max()) + 1
num_regions = int(df["TRIAL_REGION_NAME_id"].max()) + 1

agg = {TARGET_COL: "sum"}
for col in STATIC_COLS + [f"{c}_id" for c in CATEGORICAL_COLS]:
    agg[col] = "first"
for col in RAW_TIME_COV_COLS + TIME_COV_COLS:
    if col in df.columns and col not in agg:
        agg[col] = "mean"

df = (
    df.set_index(TIME_COL)
      .groupby(HOUSEHOLD_ID_COL)
      .resample(pandas_freq(FREQ))
      .agg(agg)
      .reset_index()
      .sort_values([HOUSEHOLD_ID_COL, TIME_COL])
)
add_cyclical_time_features(df, df[TIME_COL])

for col in STATIC_COLS + TIME_COV_COLS + [TARGET_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0).astype("float32")
for col in [f"{c}_id" for c in CATEGORICAL_COLS]:
    df[col] = pd.to_numeric(df[col], errors="coerce").ffill().bfill().fillna(0).astype("int64")

print("Dataframe shape:", df.shape)
print("num_stations:", num_stations, "| num_regions:", num_regions)

In [ ]:

rng = np.random.default_rng(SEED)
all_customers = np.array(sorted(df[HOUSEHOLD_ID_COL].dropna().unique()))
n_val = max(1, int(len(all_customers) * VAL_FRAC))
val_customers = set(rng.choice(all_customers, size=n_val, replace=False).tolist())
print(f"Customers total={len(all_customers)} | train={len(all_customers)-len(val_customers)} | val={len(val_customers)}")

def empty_float(*shape):
    return np.empty(shape, dtype=np.float32)

def empty_int(*shape):
    return np.empty(shape, dtype=np.int64)

y_past_parts, y_future_parts = [], []
x_time_past_parts, x_time_future_parts = [], []
x_static_parts, x_static_cat_parts = [], []
y_past_val_parts, y_future_val_parts = [], []
x_time_past_val_parts, x_time_future_val_parts = [], []
x_static_val_parts, x_static_cat_val_parts = [], []

for household_id, g in df.groupby(HOUSEHOLD_ID_COL):
    g = g.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL], keep="last")
    if g.empty:
        continue

    idx = pd.date_range(g[TIME_COL].min(), g[TIME_COL].max(), freq=pandas_freq(FREQ))
    g = g.set_index(TIME_COL).reindex(idx)
    g.index.name = TIME_COL
    g[HOUSEHOLD_ID_COL] = household_id

    for col in STATIC_COLS:
        g[col] = pd.to_numeric(g[col], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    for col in [f"{c}_id" for c in CATEGORICAL_COLS]:
        g[col] = pd.to_numeric(g[col], errors="coerce").ffill().bfill().fillna(0).astype("int64")

    add_cyclical_time_features(g, g.index)
    g[TARGET_COL] = pd.to_numeric(g[TARGET_COL], errors="coerce").fillna(0.0).astype("float32")
    for col in TIME_COV_COLS:
        g[col] = pd.to_numeric(g[col], errors="coerce").fillna(0.0).astype("float32")

    g = g.reset_index()
    if len(g) < CONTEXT_LEN + SEQ_LEN:
        continue

    y_all = g[[TARGET_COL]].to_numpy(dtype=np.float32)
    x_time_all = g[TIME_COV_COLS].to_numpy(dtype=np.float32)
    x_static_vec = g[STATIC_COLS].iloc[0].to_numpy(dtype=np.float32)
    x_cat_vec = g[[f"{c}_id" for c in CATEGORICAL_COLS]].iloc[0].to_numpy(dtype=np.int64)

    is_val = household_id in val_customers
    yp_list = y_past_val_parts if is_val else y_past_parts
    yf_list = y_future_val_parts if is_val else y_future_parts
    xtp_list = x_time_past_val_parts if is_val else x_time_past_parts
    xtf_list = x_time_future_val_parts if is_val else x_time_future_parts
    xs_list = x_static_val_parts if is_val else x_static_parts
    xc_list = x_static_cat_val_parts if is_val else x_static_cat_parts

    max_start = len(g) - CONTEXT_LEN - SEQ_LEN
    for start in range(0, max_start + 1, WINDOW_STRIDE):
        mid = start + CONTEXT_LEN
        end = mid + SEQ_LEN
        yp_list.append(y_all[start:mid])
        yf_list.append(y_all[mid:end])
        xtp_list.append(x_time_all[start:mid])
        xtf_list.append(x_time_all[mid:end])
        xs_list.append(x_static_vec)
        xc_list.append(x_cat_vec)

y_past = np.stack(y_past_parts).astype(np.float32)
y_future = np.stack(y_future_parts).astype(np.float32)
x_time_past = np.stack(x_time_past_parts).astype(np.float32)
x_time_future = np.stack(x_time_future_parts).astype(np.float32)
x_static = np.stack(x_static_parts).astype(np.float32)
static_cat_ids = np.stack(x_static_cat_parts).astype(np.int64)

y_past_val = np.stack(y_past_val_parts).astype(np.float32)
y_future_val = np.stack(y_future_val_parts).astype(np.float32)
x_time_past_val = np.stack(x_time_past_val_parts).astype(np.float32)
x_time_future_val = np.stack(x_time_future_val_parts).astype(np.float32)
x_static_val = np.stack(x_static_val_parts).astype(np.float32)
static_cat_ids_val = np.stack(x_static_cat_val_parts).astype(np.int64)

print("TRAIN y_past:", y_past.shape, "y_future:", y_future.shape, "x_time_past:", x_time_past.shape, "x_time_future:", x_time_future.shape)
print("VAL   y_past:", y_past_val.shape, "y_future:", y_future_val.shape, "x_time_past:", x_time_past_val.shape, "x_time_future:", x_time_future_val.shape)

y_scaler = MinMaxScaler(feature_range=(0, 1))
y_train_log_flat = np.concatenate([
    np.log1p(y_past).reshape(-1, 1),
    np.log1p(y_future).reshape(-1, 1),
], axis=0)
y_scaler.fit(y_train_log_flat)

# Applies the target scaler only to kWh values so generated outputs can be inverted back to physical units.
def scale_y(arr):
    return y_scaler.transform(np.log1p(arr).reshape(-1, 1)).reshape(arr.shape).astype(np.float32)

# Converts scaled model outputs back to kWh for metrics and plots.
def y_scaled_to_kwh(arr_scaled):
    arr_log = y_scaler.inverse_transform(np.asarray(arr_scaled).reshape(-1, 1)).reshape(np.asarray(arr_scaled).shape)
    return np.maximum(np.expm1(arr_log), 0.0).astype(np.float32)

y_past_scaled = scale_y(y_past)
y_future_scaled = scale_y(y_future)
y_past_scaled_val = scale_y(y_past_val)
y_future_scaled_val = scale_y(y_future_val)

c_dim = x_time_future.shape[2]
c_scaler = StandardScaler()
c_scaler.fit(np.concatenate([
    x_time_past.reshape(-1, c_dim),
    x_time_future.reshape(-1, c_dim),
], axis=0))
x_time_past_scaled = c_scaler.transform(x_time_past.reshape(-1, c_dim)).reshape(x_time_past.shape).astype(np.float32)
x_time_future_scaled = c_scaler.transform(x_time_future.reshape(-1, c_dim)).reshape(x_time_future.shape).astype(np.float32)
x_time_past_scaled_val = c_scaler.transform(x_time_past_val.reshape(-1, c_dim)).reshape(x_time_past_val.shape).astype(np.float32)
x_time_future_scaled_val = c_scaler.transform(x_time_future_val.reshape(-1, c_dim)).reshape(x_time_future_val.shape).astype(np.float32)

s_scaler = StandardScaler()
x_static_scaled = s_scaler.fit_transform(x_static).astype(np.float32)
x_static_scaled_val = s_scaler.transform(x_static_val).astype(np.float32)

# Packs past context, future target, temporal covariates, and static covariates into one training item.
class EnergyForecastDataset(Dataset):
    def __init__(self, y_past, y_future, x_time_past, x_time_future, x_static_num, x_static_cat):
        self.y_past = torch.from_numpy(y_past)
        self.y_future = torch.from_numpy(y_future)
        self.x_time_past = torch.from_numpy(x_time_past)
        self.x_time_future = torch.from_numpy(x_time_future)
        self.x_static_num = torch.from_numpy(x_static_num)
        self.x_static_cat = torch.from_numpy(x_static_cat).long()

    def __len__(self):
        return self.y_future.shape[0]

    def __getitem__(self, idx):
        return (
            self.y_past[idx],
            self.y_future[idx],
            self.x_time_past[idx],
            self.x_time_future[idx],
            self.x_static_num[idx],
            self.x_static_cat[idx],
        )

dataset = EnergyForecastDataset(
    y_past_scaled, y_future_scaled, x_time_past_scaled, x_time_future_scaled,
    x_static_scaled, static_cat_ids
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, pin_memory=torch.cuda.is_available())

y_dim = y_future_scaled.shape[2]
c_dim = x_time_future_scaled.shape[2]
s_dim = x_static_scaled.shape[1]

print("scaled ranges:", y_past_scaled.min(), y_past_scaled.max(), y_future_scaled.min(), y_future_scaled.max())
print("dims:", "y_dim", y_dim, "c_dim", c_dim, "s_dim", s_dim)

In [ ]:

# Sinusoidal positional encoding follows the Transformer paper; source: https://arxiv.org/abs/1706.03762
# Implementation reference for PyTorch Transformer/RNN sequence modelling: https://github.com/pytorch/examples/tree/main/word_language_model
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Transformer context encoder uses attention over the past context window instead of an RNN summary.
class PastContextTransformer(nn.Module):
    def __init__(self, y_dim, c_dim, hidden_dim, context_len, layers, heads, ff_dim, dropout):
        super().__init__()
        self.input_proj = nn.Linear(y_dim + c_dim, hidden_dim)
        self.pos = SinusoidalPositionalEncoding(hidden_dim, context_len)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=heads, dim_feedforward=ff_dim, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, y_past, x_time_past):
        h = self.input_proj(torch.cat([y_past, x_time_past], dim=-1))
        h = self.encoder(self.pos(h))
        return self.norm(h.mean(dim=1))
# Combines future time covariates, static numeric features, categorical embeddings, and optional past context.
class FutureConditionProjector(nn.Module):
    def __init__(self, c_dim, s_num_dim, past_dim, cond_dim, num_stations, num_regions, station_emb_dim=16, region_emb_dim=4):
        super().__init__()
        self.station_emb = nn.Embedding(num_stations, station_emb_dim)
        self.region_emb = nn.Embedding(num_regions, region_emb_dim)
        in_dim = c_dim + s_num_dim + station_emb_dim + region_emb_dim + past_dim
        self.proj = nn.Sequential(
            nn.Linear(in_dim, cond_dim),
            nn.LayerNorm(cond_dim),
            nn.Tanh(),
        )

    def forward(self, x_time_future, x_static_num, x_static_cat, past_context):
        B, T, _ = x_time_future.shape
        station_id = x_static_cat[:, 0].long()
        region_id = x_static_cat[:, 1].long()
        st = self.station_emb(station_id).unsqueeze(1).expand(B, T, -1)
        rg = self.region_emb(region_id).unsqueeze(1).expand(B, T, -1)
        xs = x_static_num.unsqueeze(1).expand(B, T, -1)
        pc = past_context.unsqueeze(1).expand(B, T, -1)
        return self.proj(torch.cat([x_time_future, xs, st, rg, pc], dim=-1))
# Transformer VAE encoder replaces recurrence with self-attention over the window; source: https://arxiv.org/abs/1706.03762
# Implementation reference for PyTorch Transformer sequence modelling: https://github.com/pytorch/examples/tree/main/word_language_model
class TransformerSequenceEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, latent_dim, seq_len, layers, heads, ff_dim, dropout):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.pos = SinusoidalPositionalEncoding(hidden_dim, seq_len)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=heads, dim_feedforward=ff_dim, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.encoder(self.pos(self.input_proj(x)))
        pooled = self.norm(h.mean(dim=1))
        return self.mu(pooled), self.logvar(pooled).clamp(-8.0, 4.0)

# Transformer VAE decoder reconstructs the window using attention plus latent conditioning; source: https://arxiv.org/abs/1706.03762
# Implementation reference for PyTorch Transformer sequence modelling: https://github.com/pytorch/examples/tree/main/word_language_model
class TransformerSequenceDecoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, y_dim, seq_len, layers, heads, ff_dim, dropout):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.pos = SinusoidalPositionalEncoding(hidden_dim, seq_len)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=heads, dim_feedforward=ff_dim, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.decoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Linear(hidden_dim, y_dim)

    def forward(self, x):
        h = self.decoder(self.pos(self.input_proj(x)))
        return self.fc(self.norm(h))


In [ ]:

# Conditional VAE forecaster with latent sampling and KL regularisation; sources: https://arxiv.org/abs/1312.6114 and https://arxiv.org/abs/2111.08095
# Implementation reference for a compact PyTorch VAE pattern: https://github.com/pytorch/examples/tree/main/vae
class ForecastTimeVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.past = PastContextTransformer(y_dim, c_dim, hidden_dim, CONTEXT_LEN, transformer_layers, transformer_heads, transformer_ff_dim, transformer_dropout)
        self.encoder = TransformerSequenceEncoder(y_dim + cond_dim, hidden_dim, latent_dim, SEQ_LEN, transformer_layers, transformer_heads, transformer_ff_dim, transformer_dropout)
        self.decoder = TransformerSequenceDecoder(latent_dim + cond_dim, hidden_dim, y_dim, SEQ_LEN, transformer_layers, transformer_heads, transformer_ff_dim, transformer_dropout)
        self.cond = FutureConditionProjector(c_dim, s_dim, hidden_dim, cond_dim, num_stations, num_regions)

    # Builds the conditional tensor reused by training, sampling, and evaluation.
    def build_cond(self, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        past_context = self.past(y_past, x_time_past)
        cond = self.cond(x_time_future, x_static_num, x_static_cat, past_context)
        return cond

    def encode(self, y_future, cond):
        return self.encoder(torch.cat([y_future, cond], dim=-1))

    def decode(self, z, cond):
        z_rep = z.unsqueeze(1).expand(z.shape[0], cond.shape[1], z.shape[-1])
        y_logits = self.decoder(torch.cat([z_rep, cond], dim=-1))
        return torch.sigmoid(y_logits), y_logits

    def forward(self, y_future, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        cond = self.build_cond(y_past, x_time_past, x_time_future, x_static_num, x_static_cat)
        mu, logvar = self.encode(y_future, cond)
        # VAE reparameterisation uses sigma=exp(0.5 logvar) then z=mu+sigma*eps; source: https://arxiv.org/abs/1312.6114
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        y_hat, y_logits = self.decode(z, cond)
        return y_hat, y_logits, mu, logvar

    @torch.no_grad()
    def sample(self, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        self.eval()
        cond = self.build_cond(y_past, x_time_past, x_time_future, x_static_num, x_static_cat)
        z = torch.randn(y_past.shape[0], latent_dim, device=y_past.device) * sample_temperature
        y_hat, _ = self.decode(z, cond)
        return y_hat.clamp(0.0, 1.0)

model_vae = ForecastTimeVAE().to(DEVICE)
opt_vae = optim.AdamW(model_vae.parameters(), lr=LR, weight_decay=1e-4)

for epoch in range(epochs_vae):
    model_vae.train()
    running_loss = running_recon = running_kl = 0.0
    # KL warmup gradually turns on VAE regularisation to reduce early posterior collapse; source: https://arxiv.org/abs/1511.06349
    kl_w = beta * min(1.0, (epoch + 1) / kl_warmup_epochs)

    for y_past_b, y_future_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b in loader:
        y_past_b = y_past_b.to(DEVICE, non_blocking=True)
        y_future_b = y_future_b.to(DEVICE, non_blocking=True)
        x_time_past_b = x_time_past_b.to(DEVICE, non_blocking=True)
        x_time_future_b = x_time_future_b.to(DEVICE, non_blocking=True)
        x_static_b = x_static_b.to(DEVICE, non_blocking=True)
        x_static_cat_b = x_static_cat_b.to(DEVICE, non_blocking=True).long()

        opt_vae.zero_grad(set_to_none=True)
        y_hat, y_logits, mu, logvar = model_vae(y_future_b, y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)
        recon = F.binary_cross_entropy_with_logits(y_logits, y_future_b)
        # Closed-form KL from diagonal Gaussian q(z|x) to N(0,I); source: https://arxiv.org/abs/1312.6114
        kl_per = 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar)
        kl = kl_per.sum(dim=1).mean()
        loss = recon + kl_w * kl
        loss.backward()
        if grad_clip:
            torch.nn.utils.clip_grad_norm_(model_vae.parameters(), grad_clip)
        opt_vae.step()

        running_loss += loss.item()
        running_recon += recon.item()
        running_kl += kl.item()

    if epoch % 10 == 0:
        print(f"[{MODEL_LABEL}] epoch {epoch:03d} | loss={running_loss/len(loader):.4f} | recon={running_recon/len(loader):.4f} | kl={running_kl/len(loader):.4f} | kl_w={kl_w:.4f}")

active_model = model_vae

@torch.no_grad()
# Generates one batch of probabilistic forecasts in scaled space before kWh inversion.
def generate_one_batch(y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b):
    return active_model.sample(y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)

In [ ]:

# Saves enough metadata with the weights to identify the exact experiment settings later.
def checkpoint_payload():
    extra = {
        "model_type": MODEL_LABEL,
        "FREQ": FREQ,
        "PRIMARY_HORIZON": PRIMARY_HORIZON,
        "SEQ_LEN": SEQ_LEN,
        "CONTEXT_LEN": CONTEXT_LEN,
        "WINDOW_STRIDE": WINDOW_STRIDE,
        "TIME_COV_COLS": TIME_COV_COLS,
        "STATIC_COLS": STATIC_COLS,
        "CATEGORICAL_COLS": CATEGORICAL_COLS,
        "y_scaler_min": y_scaler.min_.tolist(),
        "y_scaler_scale": y_scaler.scale_.tolist(),
        "num_stations": num_stations,
        "num_regions": num_regions,
        "hidden_dim": hidden_dim,
        "cond_dim": cond_dim,
        "latent_dim": latent_dim,
    }
    return extra

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_name = (
    f"{MODEL_TAG}__freq-{FREQ}__hor-{PRIMARY_HORIZON}__ctx-{CONTEXT_LEN}__seq-{SEQ_LEN}"
    f"__seed-{SEED}"
)
checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pt"

payload = {"extra": checkpoint_payload(), "model_state_dict": model_vae.state_dict(), "optimizer_state_dict": opt_vae.state_dict()}
torch.save(payload, checkpoint_path)
print("Saved:", checkpoint_path)

In [ ]:

SAMPLES = 200
EVAL_BATCH_SIZE = 64
EVAL_SEED = SEED + 123
EVAL_POOLS = ("train", "validation")
BINS = 80
CLIP_QUANTILE = None
ALPHA_PI = 0.10

# Dynamic Time Warping compares sequence shape even when peaks are slightly shifted; source: https://en.wikipedia.org/wiki/Dynamic_time_warping
def dtw_distance(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    n, m = len(a), len(b)
    dp = np.full((n + 1, m + 1), np.inf, dtype=np.float32)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return float(dp[n, m])

# Autocorrelation checks whether generated windows preserve temporal dependence; source: https://en.wikipedia.org/wiki/Autocorrelation
def autocorr(x, max_lag):
    x = np.asarray(x, dtype=np.float32)
    x = x - x.mean()
    denom = np.dot(x, x) + 1e-12
    out = [1.0]
    for lag in range(1, max_lag + 1):
        out.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.asarray(out)

def mean_acf(arr):
    max_lag = min(200, arr.shape[1] - 1)
    return np.mean([autocorr(arr[i, :, 0], max_lag) for i in range(arr.shape[0])], axis=0)

def window_feature_embed(x):
    y = x[:, :, 0]
    acf_lag = min(3, y.shape[1] - 1)
    acfs = np.stack([autocorr(row, acf_lag) for row in y], axis=0)
    return np.column_stack([
        y.mean(axis=1),
        y.std(axis=1),
        y.min(axis=1),
        y.max(axis=1),
        np.quantile(y, 0.10, axis=1),
        np.quantile(y, 0.50, axis=1),
        np.quantile(y, 0.90, axis=1),
        y.sum(axis=1),
        y.argmax(axis=1) / max(1, y.shape[1] - 1),
        acfs[:, 1] if acfs.shape[1] > 1 else np.zeros(y.shape[0]),
        acfs[:, 2] if acfs.shape[1] > 2 else np.zeros(y.shape[0]),
        acfs[:, 3] if acfs.shape[1] > 3 else np.zeros(y.shape[0]),
    ]).astype(np.float64)

def sqrtm_psd(mat):
    vals, vecs = np.linalg.eigh((mat + mat.T) / 2.0)
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)) @ vecs.T

# Feature Fréchet distance mirrors FID-style Gaussian distance on summary features; source: https://arxiv.org/abs/1706.08500
def frechet_distance(a, b):
    ma, mb = a.mean(axis=0), b.mean(axis=0)
    ca = np.cov(a, rowvar=False) + np.eye(a.shape[1]) * 1e-6
    cb = np.cov(b, rowvar=False) + np.eye(b.shape[1]) * 1e-6
    sqrt_ca = sqrtm_psd(ca)
    covmean = sqrtm_psd(sqrt_ca @ cb @ sqrt_ca)
    return float(np.sum((ma - mb) ** 2) + np.trace(ca + cb - 2 * covmean))

# Pinball loss evaluates quantile forecasts; source: https://en.wikipedia.org/wiki/Quantile_regression
def pinball(y, qhat, q):
    err = y - qhat
    return np.maximum(q * err, (q - 1) * err)

def run_eval_pool(EVAL_POOL):
    rng_eval = np.random.default_rng(EVAL_SEED + (0 if EVAL_POOL == "train" else 10_000))
    if EVAL_POOL == "train":
        n = min(512, len(y_future))
        idx = rng_eval.choice(len(y_future), size=n, replace=False)
        y_past_in = y_past_scaled[idx]
        x_time_past_in = x_time_past_scaled[idx]
        x_time_future_in = x_time_future_scaled[idx]
        x_static_in = x_static_scaled[idx]
        x_static_cat_in = static_cat_ids[idx]
        y_real_kwh = y_future[idx]
    elif EVAL_POOL == "validation":
        n = min(512, len(y_future_val))
        idx = rng_eval.choice(len(y_future_val), size=n, replace=False)
        y_past_in = y_past_scaled_val[idx]
        x_time_past_in = x_time_past_scaled_val[idx]
        x_time_future_in = x_time_future_scaled_val[idx]
        x_static_in = x_static_scaled_val[idx]
        x_static_cat_in = static_cat_ids_val[idx]
        y_real_kwh = y_future_val[idx]
    else:
        raise ValueError(EVAL_POOL)

    PLOT_N = len(y_real_kwh)
    selected_train_windows = PLOT_N if EVAL_POOL == "train" else 0
    selected_val_windows = PLOT_N if EVAL_POOL == "validation" else 0
    DTW_N = min(128, PLOT_N)
    print(f"eval_pool={EVAL_POOL} | windows={PLOT_N} | samples={SAMPLES}")

    y_fake_scaled_samples = []
    active_model.eval()
    with torch.no_grad():
        for s in range(SAMPLES):
            if s % 10 == 0:
                print(f"Generating {MODEL_LABEL} sample {s+1}/{SAMPLES}")
            batches = []
            for start in range(0, PLOT_N, EVAL_BATCH_SIZE):
                end = min(start + EVAL_BATCH_SIZE, PLOT_N)
                ypb = torch.from_numpy(y_past_in[start:end]).to(DEVICE)
                xtpb = torch.from_numpy(x_time_past_in[start:end]).to(DEVICE)
                xtfb = torch.from_numpy(x_time_future_in[start:end]).to(DEVICE)
                xsb = torch.from_numpy(x_static_in[start:end]).to(DEVICE)
                xcb = torch.from_numpy(x_static_cat_in[start:end]).to(DEVICE).long()
                batches.append(generate_one_batch(ypb, xtpb, xtfb, xsb, xcb).cpu().numpy())
            y_fake_scaled_samples.append(np.concatenate(batches, axis=0))

    y_fake_scaled_samples = np.stack(y_fake_scaled_samples, axis=0)
    y_fake_kwh_samples = np.stack([y_scaled_to_kwh(y_fake_scaled_samples[s]) for s in range(SAMPLES)], axis=0)
    y_fake_kwh_point = np.median(y_fake_kwh_samples, axis=0)
    y_fake_kwh_mean = np.mean(y_fake_kwh_samples, axis=0)

    MAE_median = float(np.mean(np.abs(y_fake_kwh_point - y_real_kwh)))
    RMSE_mean = float(np.sqrt(np.mean((y_fake_kwh_mean - y_real_kwh) ** 2)))
    PeakMAE = float(np.mean(np.abs(y_fake_kwh_point.max(axis=1) - y_real_kwh.max(axis=1))))

    real_flat = y_real_kwh.reshape(-1)
    fake_flat = y_fake_kwh_point.reshape(-1)
    lo, hi = float(real_flat.min()), float(real_flat.max())
    if hi <= lo:
        hi = lo + 1e-6
    edges = np.linspace(lo, hi, BINS + 1)
    p, _ = np.histogram(real_flat, bins=edges, density=True)
    q, _ = np.histogram(fake_flat, bins=edges, density=True)
    eps = 1e-8
    p = (p + eps) / (p + eps).sum()
    q = (q + eps) / (q + eps).sum()
    # KL histogram estimates marginal distribution mismatch from binned real/generated kWh; source: https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence
    KL_hist = float(np.sum(p * np.log(p / q)))

    dtw_idxs = rng_eval.choice(PLOT_N, size=DTW_N, replace=False)
    DTW_mean = float(np.mean([dtw_distance(y_real_kwh[i, :, 0], y_fake_kwh_point[i, :, 0]) for i in dtw_idxs]))

    FTSD = frechet_distance(window_feature_embed(y_real_kwh), window_feature_embed(y_fake_kwh_point))

    X = y_fake_kwh_samples[:, :, :, 0]
    Y = y_real_kwh[:, :, 0]
    q10 = np.quantile(X, 0.05, axis=0)
    q50 = np.quantile(X, 0.50, axis=0)
    q90 = np.quantile(X, 0.95, axis=0)
    # CRPS is approximated from quantile pinball losses; source: https://en.wikipedia.org/wiki/Continuous_ranked_probability_score
    CRPS = float(2.0 * np.mean([
        np.mean(pinball(Y, q10, 0.05)),
        np.mean(pinball(Y, q50, 0.50)),
        np.mean(pinball(Y, q90, 0.95)),
    ]))
    # QuantileLoss averages pinball losses at the requested forecast quantiles; source: https://en.wikipedia.org/wiki/Quantile_regression
    QuantileLoss = float(np.mean([
        np.mean(pinball(Y, q10, 0.05)),
        np.mean(pinball(Y, q50, 0.50)),
        np.mean(pinball(Y, q90, 0.95)),
    ]))
    width = q90 - q10
    below = Y < q10
    above = Y > q90
    # Winkler interval score rewards narrow prediction intervals but penalises missed coverage; source: https://epiforecasts.io/scoringutils/articles/scoring-rules.html
    Winkler = float(np.mean(width + (2 / ALPHA_PI) * (q10 - Y) * below + (2 / ALPHA_PI) * (Y - q90) * above))
    Coverage = float(np.mean((Y >= q10) & (Y <= q90)))
    ACF_error = float(np.mean(np.abs(mean_acf(y_real_kwh) - mean_acf(y_fake_kwh_point))))

    scale = float(np.mean(y_real_kwh) + 1e-6)
    peak_scale = float(np.mean(y_real_kwh.max(axis=1)) + 1e-6)
    # CompositeScore is a project-specific tuning heuristic, not a standard literature metric.
    CompositeScore = float(np.mean([
        MAE_median / scale,
        RMSE_mean / scale,
        PeakMAE / peak_scale,
        CRPS / scale,
        QuantileLoss / scale,
        Winkler / scale,
        ACF_error,
        KL_hist,
        DTW_mean / (SEQ_LEN * scale),
        min(FTSD, 1e6) / (1.0 + min(FTSD, 1e6)),
    ]))

    corr_real = np.nan
    corr_fake = np.nan
    if "Temperature" in TIME_COV_COLS:
        temp = x_time_future_in[:, :, TIME_COV_COLS.index("Temperature")].reshape(-1)
        real_y = y_real_kwh.reshape(-1)
        fake_y = y_fake_kwh_point.reshape(-1)
        def corr(a, b):
            a = np.asarray(a)
            b = np.asarray(b)
            a = a - a.mean()
            b = b - b.mean()
            return float(np.dot(a, b) / (np.sqrt(np.dot(a, a)) * np.sqrt(np.dot(b, b)) + 1e-12))
        corr_real = corr(temp, real_y)
        corr_fake = corr(temp, fake_y)

    print("=" * 60)
    print(f"[{MODEL_LABEL}] forecast eval | pool={EVAL_POOL} | freq={FREQ} | horizon={PRIMARY_HORIZON} | context={CONTEXT_LEN} | seq_len={SEQ_LEN}")
    print(f"KL_hist={KL_hist:.6f} | DTW_mean={DTW_mean:.6f} | FTSD={FTSD:.6f}")
    print(f"MAE={MAE_median:.6f} | RMSE={RMSE_mean:.6f} | PeakMAE={PeakMAE:.6f}")
    print(f"CRPS={CRPS:.6f} | QuantileLoss={QuantileLoss:.6f} | Winkler={Winkler:.6f} | Coverage={Coverage:.4f}")
    print(f"ACF_error={ACF_error:.6f} | CompositeScore={CompositeScore:.6f}")

    return locals()

eval_results = {}
for _pool in EVAL_POOLS:
    eval_results[_pool] = run_eval_pool(_pool)

In [ ]:

import importlib.util

_project_root = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "models" / "evaluation_export.py").exists()
)
_export_helper_path = _project_root / "models" / "evaluation_export.py"
_spec = importlib.util.spec_from_file_location("evaluation_export", _export_helper_path)
_evaluation_export = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_evaluation_export)
export_common_evaluation = _evaluation_export.export_common_evaluation

for eval_pool, namespace in eval_results.items():
    namespace["CHECKPOINT_PATH"] = str(checkpoint_path)
    namespace["CHECKPOINT_NAME"] = checkpoint_name
    export_dir, metrics_path, plot_paths = export_common_evaluation(
        namespace,
        model_label=MODEL_LABEL,
        checkpoint_name=checkpoint_name,
        output_root=_project_root / "evaluation_exports" / f"{eval_pool}_evaluation_exports",
    )
    print(f"Exported {eval_pool}: {export_dir}")
    print(f"Metrics: {metrics_path}")
    print(f"Plots: {len(plot_paths)}")